# PokeChess Demo

This notebook demonstrates the PokeChess game engine: board layout, piece movement, attacks, damage mechanics, and game flow. No bot logic is required — all moves are chosen manually.

Run the cells top to bottom. The board is printed as ASCII after each action.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

from engine import GameState, Piece, PieceType, PieceStats, PokemonType, Team, Item, PIECE_STATS
from engine.state import KING_TYPES, PAWN_TYPES
from engine.moves import get_legal_moves, Move, ActionType
from engine.rules import apply_move, is_terminal, hp_winner

print('Engine imported successfully.')

## Board Rendering

We'll render the board as a compact ASCII grid. Uppercase = RED, lowercase = BLUE.

In [ ]:
_ABBREV = {
    PieceType.SQUIRTLE:   'Sq',
    PieceType.CHARMANDER: 'Ch',
    PieceType.BULBASAUR:  'Bu',
    PieceType.MEW:        'Mw',
    PieceType.POKEBALL:   'Pb',
    PieceType.MASTERBALL: 'Mb',
    PieceType.PIKACHU:    'Pi',
    PieceType.RAICHU:     'Ra',
    PieceType.EEVEE:      'Ee',
    PieceType.VAPOREON:   'Va',
    PieceType.FLAREON:    'Fl',
    PieceType.LEAFEON:    'Le',
    PieceType.JOLTEON:    'Jo',
    PieceType.ESPEON:     'Es',
}

def render(state: GameState) -> None:
    """Print the board. Uppercase = RED, lowercase = BLUE."""
    turn_team = state.active_player.name
    print(f"Turn {state.turn_number}  |  Active: {turn_team}")
    print('   ' + '  '.join(str(c) for c in range(8)))
    print('  +' + '---+' * 8)
    for r in range(7, -1, -1):
        row_str = f"{r} |"
        for c in range(8):
            p = state.board[r][c]
            if p is None:
                row_str += ' . |'
            else:
                abbr = _ABBREV[p.piece_type]
                cell = abbr if p.team == Team.RED else abbr.lower()
                row_str += f'{cell}|'
        print(row_str)
        print('  +' + '---+' * 8)


def piece_summary(state: GameState, team: Team) -> None:
    """Print HP summary for all pieces of a team."""
    print(f"\n{team.name} pieces:")
    for r in range(8):
        for c in range(8):
            p = state.board[r][c]
            if p is not None and p.team == team:
                max_hp = PIECE_STATS[p.piece_type].max_hp
                bar = '#' * (p.current_hp // 20) + '.' * ((max_hp - p.current_hp) // 20)
                print(f"  ({r},{c}) {p.piece_type.name:<12} HP {p.current_hp:>3}/{max_hp:<3}  [{bar}]")

print('Rendering helpers ready.')

## Section 1 — Starting Board

Create a fresh game and inspect the initial layout.

In [ ]:
state = GameState.new_game()
render(state)
piece_summary(state, Team.RED)
piece_summary(state, Team.BLUE)

## Section 2 — Legal Move Inspection

See which moves are available for a specific piece.

In [ ]:
def moves_from(state, row, col):
    return [m for m in get_legal_moves(state) if m.piece_row == row and m.piece_col == col]


def print_moves(moves):
    by_type = {}
    for m in moves:
        by_type.setdefault(m.action_type.name, []).append((m.target_row, m.target_col))
    for at, targets in sorted(by_type.items()):
        print(f"  {at}: {targets}")


# Red Pokeball at (1,0) — pawn on row 1
print("Red Pokeball at (1,0):")
print_moves(moves_from(state, 1, 0))

# Red Pikachu at (0,4) — king
print("\nRed Pikachu at (0,4):")
print_moves(moves_from(state, 0, 4))

## Section 3 — Move Execution: Advancing Pawns

Advance a Red Pokeball two squares forward, then Blue responds.

In [ ]:
def apply(state, row, col, action_type, target_row, target_col, move_slot=None):
    """Convenience: apply the first matching move and return the new state."""
    move = Move(
        piece_row=row, piece_col=col,
        action_type=action_type,
        target_row=target_row, target_col=target_col,
        move_slot=move_slot,
    )
    outcomes = apply_move(state, move)
    # For stochastic moves, pick the first outcome (capture branch).
    return outcomes[0][0]


# Red Pokeball at (1,3) advances 2 squares to (3,3)
state = apply(state, 1, 3, ActionType.MOVE, 3, 3)
# Blue Pokeball at (6,3) advances 2 squares to (4,3)
state = apply(state, 6, 3, ActionType.MOVE, 4, 3)

render(state)

## Section 4 — Piece Movement: Squirtle (Rook)

Open a column so Squirtle can slide, then move it out.

In [ ]:
# Red: advance Pokeball at (1,0) one square forward
state = apply(state, 1, 0, ActionType.MOVE, 2, 0)
# Blue: advance Pokeball at (6,0) one square
state = apply(state, 6, 0, ActionType.MOVE, 5, 0)

# Red Squirtle at (0,0) can now slide forward
print("Red Squirtle at (0,0) moves:")
print_moves(moves_from(state, 0, 0))
state = apply(state, 0, 0, ActionType.MOVE, 2, 0)  # Wait — (2,0) is occupied
render(state)

> The Squirtle can't reach (2,0) because the Pokeball is there — sliding pieces are blocked. Let's advance the pawn further and try again.

In [ ]:
# Rewind: restart the demo from a clean state that has room to maneuver.
state = GameState.new_game()

# Clear the Pokeball in front of Squirtle so it can slide.
state.board[1][0] = None   # remove Red Pokeball manually for demo purposes

print("Red Squirtle at (0,0) legal moves after clearing column:")
print_moves(moves_from(state, 0, 0))

# Slide Squirtle forward to row 4
state = apply(state, 0, 0, ActionType.MOVE, 4, 0)
# Blue: move a pokeball (so it's Blue's valid turn)
state = apply(state, 6, 1, ActionType.MOVE, 5, 1)

render(state)

## Section 5 — Combat: Damage and KO

Set up a direct confrontation and observe damage resolution.

In [ ]:
from engine.state import MATCHUP

# Quick damage preview helper
def damage_preview(attacker_type: PieceType, defender_type: PieceType, move_slot=None):
    from engine.rules import _calc_damage  # internal, for demo only
    atk = Piece.create(attacker_type, Team.RED, 0, 0)
    dfn = Piece.create(defender_type, Team.BLUE, 0, 1)
    atk.held_item = Item.NONE  # strip item to show base type multiplier only
    dmg = _calc_damage(atk, dfn, move_slot)
    print(f"  {attacker_type.name} → {defender_type.name}: {dmg} damage (no item)")

print("Damage preview (no held items):")
damage_preview(PieceType.SQUIRTLE, PieceType.CHARMANDER)   # Water > Fire = 2×
damage_preview(PieceType.SQUIRTLE, PieceType.BULBASAUR)    # Water < Grass = 0.5×
damage_preview(PieceType.SQUIRTLE, PieceType.SQUIRTLE)     # Water vs Water = 0.5×
damage_preview(PieceType.CHARMANDER, PieceType.BULBASAUR)  # Fire > Grass = 2×
damage_preview(PieceType.MEW, PieceType.SQUIRTLE, 3)       # Psychic slot 3 = 200 → KO

In [ ]:
# Build a confrontation board: Red Squirtle adjacent to Blue Charmander
from engine.state import ForesightEffect

def empty_state():
    board = [[None] * 8 for _ in range(8)]
    return GameState(
        board=board,
        active_player=Team.RED,
        turn_number=1,
        pending_foresight={Team.RED: None, Team.BLUE: None},
    )


combat = empty_state()
red_sq = Piece.create(PieceType.SQUIRTLE, Team.RED, 4, 3)
red_sq.held_item = Item.NONE  # no item for clean numbers
blue_ch = Piece.create(PieceType.CHARMANDER, Team.BLUE, 4, 4)
blue_ch.held_item = Item.NONE
# Also place kings so is_terminal doesn't trigger immediately
combat.board[0][4] = Piece.create(PieceType.PIKACHU, Team.RED, 0, 4)
combat.board[7][4] = Piece.create(PieceType.EEVEE, Team.BLUE, 7, 4)
combat.board[4][3] = red_sq
combat.board[4][4] = blue_ch

print("Before attack:")
render(combat)
print(f"  Blue Charmander HP: {blue_ch.current_hp}/{PIECE_STATS[PieceType.CHARMANDER].max_hp}")

# Squirtle attacks Charmander (Water 2× Fire = 160 damage)
combat2 = apply(combat, 4, 3, ActionType.ATTACK, 4, 4)
print("\nAfter Squirtle attacks Charmander (expected 160 damage):")
piece_at = combat2.board[4][4]
if piece_at and piece_at.team == Team.BLUE:
    print(f"  Blue Charmander HP: {piece_at.current_hp}/{PIECE_STATS[PieceType.CHARMANDER].max_hp}")
else:
    print("  Charmander KO'd — Squirtle occupies (4,4)")
render(combat2)

## Section 6 — Held Items (Evolution Only)

Items are used solely to gate evolution — they have no effect on damage. A Squirtle with WATERSTONE deals the same damage as one without.

In [ ]:
print("Squirtle → Charmander (Water 2× Fire), with vs without WATERSTONE:")

from engine.rules import _calc_damage

def damage_check(attacker_type, defender_type, item):
    atk = Piece.create(attacker_type, Team.RED, 0, 0)
    dfn = Piece.create(defender_type, Team.BLUE, 0, 1)
    atk.held_item = item
    return _calc_damage(atk, dfn)

no_item = damage_check(PieceType.SQUIRTLE, PieceType.CHARMANDER, Item.NONE)
with_stone = damage_check(PieceType.SQUIRTLE, PieceType.CHARMANDER, Item.WATERSTONE)
print(f"  No item:    {no_item} dmg")
print(f"  WATERSTONE: {with_stone} dmg  (same — items only unlock evolutions)")

## Section 7 — Stochastic Pokeball Capture

Pokeball attacks return two outcomes at p=0.5 each. Pikachu and Raichu are immune.

In [ ]:
pb_demo = empty_state()
pb_demo.board[0][4] = Piece.create(PieceType.PIKACHU, Team.RED, 0, 4)
pb_demo.board[7][4] = Piece.create(PieceType.EEVEE, Team.BLUE, 7, 4)
red_pb = Piece.create(PieceType.POKEBALL, Team.RED, 3, 3)
blue_sq = Piece.create(PieceType.SQUIRTLE, Team.BLUE, 3, 4)
pb_demo.board[3][3] = red_pb
pb_demo.board[3][4] = blue_sq

move = Move(piece_row=3, piece_col=3, action_type=ActionType.ATTACK,
            target_row=3, target_col=4)
outcomes = apply_move(pb_demo, move)

print(f"Number of outcomes: {len(outcomes)}")
for i, (ns, prob) in enumerate(outcomes):
    target_after = ns.board[3][4]
    if target_after and target_after.team == Team.BLUE:
        result = f"Squirtle still alive (capture failed)"
    else:
        result = f"Squirtle captured — Pokeball at (3,4)"
    print(f"  Outcome {i}: p={prob}  →  {result}")

# Pikachu immunity
print()
blue_pika = Piece.create(PieceType.PIKACHU, Team.BLUE, 3, 5)
pb_demo.board[3][5] = blue_pika
pika_move = Move(piece_row=3, piece_col=3, action_type=ActionType.ATTACK,
                 target_row=3, target_col=5)
pika_outcomes = apply_move(pb_demo, pika_move)
print(f"Pokeball vs Pikachu — outcomes: {len(pika_outcomes)} (deterministic failure)")
for ns, prob in pika_outcomes:
    still_there = ns.board[3][5] is not None and ns.board[3][5].piece_type == PieceType.PIKACHU
    print(f"  p={prob}  Pikachu still alive: {still_there}")

## Section 8 — Foresight

Mew schedules a Foresight attack that resolves two half-moves later.

In [ ]:
fs_demo = empty_state()
fs_demo.board[0][4] = Piece.create(PieceType.PIKACHU, Team.RED, 0, 4)
fs_demo.board[7][4] = Piece.create(PieceType.EEVEE, Team.BLUE, 7, 4)
red_mew = Piece.create(PieceType.MEW, Team.RED, 3, 3)
blue_sq = Piece.create(PieceType.SQUIRTLE, Team.BLUE, 3, 5)
fs_demo.board[3][3] = red_mew
fs_demo.board[3][5] = blue_sq

print(f"Before Foresight — Blue Squirtle HP: {blue_sq.current_hp}")

# Turn 1: Red Mew casts Foresight on (3,5)
fs_move = Move(piece_row=3, piece_col=3, action_type=ActionType.FORESIGHT,
               target_row=3, target_col=5)
[(s1, _)] = apply_move(fs_demo, fs_move)
fx = s1.pending_foresight[Team.RED]
print(f"After Foresight cast (turn {s1.turn_number}): resolves_on_turn={fx.resolves_on_turn}, damage={fx.damage}")

# Turn 2: Blue makes a pass move (move king in place — just advance the turn)
blue_king = s1.board[7][4]
noop = Move(piece_row=7, piece_col=4, action_type=ActionType.MOVE, target_row=6, target_col=4)
[(s2, _)] = apply_move(s1, noop)
print(f"After Blue's move (turn {s2.turn_number}): Foresight pending for RED = {s2.pending_foresight[Team.RED]}")

# Turn 3: Red makes any move — Foresight resolves FIRST at start of Red's turn
red_king_move = Move(piece_row=0, piece_col=4, action_type=ActionType.MOVE, target_row=0, target_col=3)
[(s3, _)] = apply_move(s2, red_king_move)
target_after = s3.board[3][5]
if target_after is None:
    print(f"Turn {s3.turn_number}: Foresight KO'd Blue Squirtle!")
else:
    print(f"Turn {s3.turn_number}: Blue Squirtle HP after Foresight = {target_after.current_hp}")
print(f"Foresight cleared: {s3.pending_foresight[Team.RED] is None}")

## Section 9 — King Evolution

Pikachu evolves into Raichu (costs a turn). Eevee evolves using a held stone.

In [ ]:
evo_demo = empty_state()
pika = Piece.create(PieceType.PIKACHU, Team.RED, 4, 4)
eevee = Piece.create(PieceType.EEVEE, Team.BLUE, 4, 3)
eevee.held_item = Item.WATERSTONE  # Eevee → Vaporeon
evo_demo.board[4][4] = pika
evo_demo.board[4][3] = eevee

print(f"Before: RED={pika.piece_type.name} (HP {pika.current_hp}), BLUE={eevee.piece_type.name} (item={eevee.held_item.name})")

# Red: Pikachu evolves (EVOLVE action targeting own square)
evolve_pika = Move(piece_row=4, piece_col=4, action_type=ActionType.EVOLVE,
                   target_row=4, target_col=4)
[(s1, _)] = apply_move(evo_demo, evolve_pika)
raichu = s1.board[4][4]
print(f"After RED evolve: {raichu.piece_type.name} (HP {raichu.current_hp}/{PIECE_STATS[PieceType.RAICHU].max_hp}), item={raichu.held_item.name}")

# Blue: Eevee evolves into Vaporeon
evolve_eevee = Move(piece_row=4, piece_col=3, action_type=ActionType.EVOLVE,
                    target_row=4, target_col=3, move_slot=0)  # slot 0 = Vaporeon
[(s2, _)] = apply_move(s1, evolve_eevee)
vaporeon = s2.board[4][3]
print(f"After BLUE evolve: {vaporeon.piece_type.name} (HP {vaporeon.current_hp}/{PIECE_STATS[PieceType.VAPOREON].max_hp}), item={vaporeon.held_item.name}")

## Section 10 — Win Condition and HP Tiebreaker

In [ ]:
win_demo = empty_state()
win_demo.board[0][4] = Piece.create(PieceType.PIKACHU, Team.RED, 0, 4)
win_demo.board[7][4] = Piece.create(PieceType.EEVEE, Team.BLUE, 7, 4)

done, winner = is_terminal(win_demo)
print(f"Both kings alive → terminal={done}, winner={winner}")

# Remove Blue's king
win_demo.board[7][4] = None
done, winner = is_terminal(win_demo)
print(f"Blue king removed → terminal={done}, winner={winner.name if winner else None}")

# HP tiebreaker demo
hp_demo = empty_state()
red_sq = Piece.create(PieceType.SQUIRTLE, Team.RED, 3, 3)
blue_sq = Piece.create(PieceType.SQUIRTLE, Team.BLUE, 5, 5)
red_sq.current_hp = 150
blue_sq.current_hp = 120
hp_demo.board[3][3] = red_sq
hp_demo.board[5][5] = blue_sq
result = hp_winner(hp_demo)
print(f"\nHP tiebreaker (RED 150 vs BLUE 120) → {result.name if result else None}")

## Section 11 — Mini Game: 10 Random Moves

Play a short game from the starting position using random move selection to demonstrate full game flow.

In [ ]:
import random
random.seed(42)

game = GameState.new_game()
print("Starting position:")
render(game)

for turn in range(1, 11):
    done, winner = is_terminal(game)
    if done:
        print(f"\nGame over! Winner: {winner.name if winner else 'DRAW'}")
        break

    moves = get_legal_moves(game)
    if not moves:
        print(f"Turn {turn}: {game.active_player.name} has no legal moves.")
        break

    move = random.choice(moves)
    outcomes = apply_move(game, move)
    game = outcomes[0][0] if len(outcomes) == 1 else random.choices(
        [s for s, _ in outcomes], weights=[p for _, p in outcomes])[0]

    print(f"\nTurn {turn}: {('RED' if game.active_player == Team.BLUE else 'BLUE')} plays "
          f"{move.action_type.name} ({move.piece_row},{move.piece_col})→({move.target_row},{move.target_col})")

print("\nFinal state after 10 moves:")
render(game)
tiebreak = hp_winner(game)
print(f"HP tiebreaker (if needed): {tiebreak.name if tiebreak else 'TIE'}")